In [1]:
import pandas as pd
import sqlite3
import pickle
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from imblearn.over_sampling import SMOTE

# --- 1. CONNECT TO DATABASE (SQL Extraction) ---
# We go 'up' one level (..) to find the data folder
conn = sqlite3.connect('../data/credit_risk.db') 
query = "SELECT * FROM loans"
df = pd.read_sql(query, conn)
conn.close()

print("✅ Data loaded from SQL Database.")

# --- 2. PREPROCESSING ---
# Convert 'Risk' to binary: good=0, bad=1 (Target Variable)
df['Risk_Code'] = df['Risk'].map({'good': 0, 'bad': 1})

# Select Features (X) and Target (y)
# For this MVP, we use the 3 numeric columns to keep it simple
features = ['Age', 'Credit amount', 'Duration']
X = df[features]
y = df['Risk_Code']

# --- 3. SPLIT & SMOTE (Balancing the Data) ---
# Split first to avoid data leakage
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Apply SMOTE to generate synthetic "Bad" loans for training
smote = SMOTE(random_state=42)
X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)

print(f"Original Training Count: {len(X_train)}")
print(f"Balanced Training Count: {len(X_train_bal)} (SMOTE Applied)")

# --- 4. TRAIN THE MODEL ---
model = LogisticRegression()
model.fit(X_train_bal, y_train_bal)

# Test Accuracy
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"📊 Model Test Accuracy: {accuracy:.2%}")

# --- 5. SAVE THE MODEL (The 'Pickle') ---
# We save it into the /data folder so the App can find it easily
with open('../data/credit_model.pkl', 'wb') as f:
    pickle.dump(model, f)

print("✅ Model saved as 'credit_model.pkl' in the data folder.")

✅ Data loaded from SQL Database.
Original Training Count: 800
Balanced Training Count: 1100 (SMOTE Applied)
📊 Model Test Accuracy: 57.50%
✅ Model saved as 'credit_model.pkl' in the data folder.


New Code

In [3]:
# analysis.ipynb logic
import pandas as pd
import sqlite3
import pickle
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from imblearn.over_sampling import SMOTE

# 1. Load & Encode
conn = sqlite3.connect('../data/credit_risk.db')
df = pd.read_sql("SELECT * FROM loans", conn)
conn.close()

# Convert Categorical to Numbers (One-Hot Encoding)
df_encoded = pd.get_dummies(df, columns=['Sex', 'Purpose'], drop_first=True)
df_encoded['Risk_Code'] = df_encoded['Risk'].map({'good': 0, 'bad': 1})

X = df_encoded.drop(['Risk', 'Risk_Code'], axis=1)
y = df_encoded['Risk_Code']

# 2. Split & SMOTE (Address Class Imbalance)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
smote = SMOTE(random_state=42)
X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)

# 3. XGBoost Model
model = xgb.XGBClassifier(n_estimators=100, learning_rate=0.05, max_depth=6)
model.fit(X_train_bal, y_train_bal)

# 4. QUANTIFY BUSINESS IMPACT (The 'Beast' part)
y_pred = model.predict(X_test)
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
# Logic: A missed 'bad' loan costs $10k. A lost 'good' customer costs $1k in interest.
impact = (fn * 10000) + (fp * 1000)
print(f"📉 Total Opportunity Loss with this model: ${impact:,}")

# 5. Save Everything
with open('../data/credit_model.pkl', 'wb') as f:
    pickle.dump(model, f)
with open('../data/feature_columns.pkl', 'wb') as f:
    pickle.dump(X.columns.tolist(), f)

📉 Total Opportunity Loss with this model: $688,000
